In [1]:
!pip install kagglehub pandas

# Rain in Australia

## Motivation and Question

## Target Variable and Class Labels

## Data Source, Observations, and Main Features

### Import Dataset

In [2]:
import kagglehub
from pathlib import Path

# Download latest version
handle = "jsphyg/weather-dataset-rattle-package"
path = "weatherAUS.csv"
if not Path(path).exists():
    print("Downloading from Kaggle")
    path = kagglehub.dataset_download(
        "jsphyg/weather-dataset-rattle-package",
        "weatherAUS.csv",
        output_dir="."
    )

print("Path to dataset files:", path)

Path to dataset files: weatherAUS.csv


/Users/ben/Docs/College/Waseda/Intermediate Data Science/ids_pbl2_rain-in-australia/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd

df = pd.read_csv(path)

## Cleaning and Preprocessing

* Handling null values should happend before enrichment.
    * Except some enrichment should happend before getting rid of null values. For example, consecutive rain days
* Enrichment should happen before EDA
* EDA should happen before preprocessing
* Some amount of EDA should handle any imputed (estimated and added in) null values, if we're using any
    * E.g. Box plot with and without imputed values

* Handle categorical values ([ordinal][sklearn.preprocessing.OrdinalEncoder], [one-hot][sklearn.preprocessing.OneHotEncoder], [target][sklearn.preprocessing.TargetEncoder])
* [Standardize scalers][sklearn.preprocessing.StandardScaler]?
* Handle null values
    * Delete: simply drop any rows/columns with missing values, especially if they are few and missingness is random
    * Impute: replace with estimated values
        * with mean, median, or mode
        * using forward/backward fill - duplicating in the previous/next seen value...
    * Interpolate from nearby values...

[sklearn.preprocessing.OrdinalEncoder]: https://scikit-learn.org/1.9/modules/generated/sklearn.preprocessing.OrdinalEncoder.html#sklearn.preprocessing.OrdinalEncoder
[sklearn.preprocessing.OneHotEncoder]: https://scikit-learn.org/1.9/modules/generated/sklearn.preprocessing.OneHotEncoder.html#sklearn.preprocessing.OneHotEncoder
[sklearn.preprocessing.TargetEncoder]: https://scikit-learn.org/1.9/modules/generated/sklearn.preprocessing.TargetEncoder.html#sklearn.preprocessing.TargetEncoder
[sklearn.preprocessing.StandardScaler]: https://scikit-learn.org/1.9/modules/generated/sklearn.preprocessing.StandardScaler.html

According to the [Australian Government's Bureau of Meterology](https://www.bom.gov.au/climate/dwo/IDCJDW0000.shtml), "From time to time, observations will not be available, for a variety of reasons. Sometimes when the daily maximum and minimum temperatures, rainfall or evaporation are missing, the next value given has been accumulated over several days rather than the normal one day. It is very difficult for an automatic system to detect this reliably, so caution is advised."

In [37]:
print(f"Number of columns with a significant amount of missing data: {((df.isnull().sum() / df.shape[0]) > 0.05).sum()} / {df.shape[1]} = {((df.isnull().sum() / df.shape[0]) > 0.05).sum() / df.shape[1]}")
print(f"Number of rows with at least one missing feature: {(df.isnull().sum(axis=1) > 0).sum()} / {df.shape[0]} = {(df.isnull().sum(axis=1) > 0).sum() / df.shape[0]}")
print(f"Number of rows with more than one missing feature: {(df.isnull().sum(axis=1) > 2).sum()} / {df.shape[0]} = {(df.isnull().sum(axis=1) > 2).sum() / df.shape[0]}")

Number of columns with a significant amount of missing data: 10 / 34 = 0.29411764705882354
Number of rows with at least one missing feature: 89315 / 145460 = 0.6140175993400248
Number of rows with more than one missing feature: 60168 / 145460 = 0.41363948851918053


### Enrich Dataset
- TempRange — MaxTemp minus MinTemp
- MonsoonSeason — location-aware wet/monsoon flag (Yes/No)
- WindSpeedDiff — WindSpeed3pm minus WindSpeed9am (signed)
- HumidityDiff — Humidity3pm minus Humidity9am (signed)
- TempDiff9am3pm — Temp3pm minus Temp9am (signed)
- PressureDiff — Pressure3pm minus Pressure9am (signed)
- DewPoint9am — dew point from 9am temp and humidity
- Season — Southern Hemisphere season from date
- DaysSinceRain — days since last rain per location
- ConsecutiveRainDays — consecutive rainy days before current day
- LaNina — monthly La Niña flag from NOAA NINO3.4 anomaly

In [5]:
from enrich_weather import enrich
df = enrich(df)

In [7]:
print("Number of columns with a significant amount of missing data", ((df.isnull().sum() / df.shape[0]) > 0.05).sum())

Number of columns with a significant amount of missing data 10


## Exploratory Data Analysis
e.g. Description: columns, data-types, null values, counts, mean, standard deviation, box plot: min, 25%, 50%, 75% max; Distribution: target, target vs. key features, correlation matrix; target class balance

## Train-Test Split

## Simple Model

## Interpretable Model
Logistic regression or decision tree

## Improved Model
Random forest, boosting, other ensemble

## Evaluation
With confucion matrix and suitable classification matrix

## Limitations

### False Positives and Negatives

## References